# Day 50 — Time-series modeling and forecast evaluation
Objectives:
- Split time-ordered data without future leakage.
- Fit a small ARIMA model with pmdarima.
- Compare it with a naive forecast using the same horizon and metric.
- Treat more complex forecasting families as hypotheses to validate, not automatic upgrades.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
idx = pd.date_range('2020-01-01', periods=300, freq='D')
trend = np.linspace(0,10,300)
season = 2*np.sin(2*np.pi*np.arange(300)/30)
noise = rng.normal(scale=1.0, size=300)
y = 20 + trend + season + noise
ts = pd.Series(y, index=idx, name='y')
ts.plot(figsize=(8,3)); plt.title('Synthetic series'); plt.show()


In [ ]:
# ARIMA with pmdarima
from pmdarima import auto_arima
train = ts.iloc[:-30]
test = ts.iloc[-30:]
model = auto_arima(train, seasonal=True, m=30, trace=False, suppress_warnings=True)
model.summary()
pred = model.predict(n_periods=len(test))
plt.plot(train.index, train.values, label='train')
plt.plot(test.index, test.values, label='test')
plt.plot(test.index, pred, label='forecast')
plt.legend(); plt.show()
from sklearn.metrics import mean_absolute_error
mean_absolute_error(test.values, pred)


## Model choice and baselines
Seasonal additive models, tree models with lag features, and neural sequence models can all be useful in the right setting. They also add assumptions and operational cost. Compare every candidate against a simple last-value or seasonal-naive forecast on a forward time split.

## Learner exercises and progressive hints

1. Fit `auto_arima` with `m=7` and `m=30`; compare mean absolute error on the
   same test window.
2. Create last-value and 30-day seasonal-naive forecasts and compare them with
   ARIMA.
3. Difference the training series and inspect autocorrelation without using
   test-period observations.

### Progressive hints

1. Change only the seasonal period. Keep the split, horizon, and MAE calculation
   fixed, and bound the search if runtime is high.
2. Build each baseline solely from `train`; verify predictions have the same
   index/length as `test`.
3. Call `train.diff().dropna()` before plotting autocorrelation. The test values
   should not appear anywhere in transformation fitting.

The reference solution extends the lesson with `TimeSeriesSplit`, shifted
rolling features, and a seasonal-naive evaluation. It uses a different
synthetic weekly series to reinforce the method rather than mirror the notebook.

### Additional mastery practice

Evaluate forecasts with forward-only information, multiple origins, meaningful baselines, and timestamp/data-quality checks before adding model complexity.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Rolling-origin evaluation:** Implement at least four expanding-window forecast origins with a fixed horizon. Compare seasonal-naive and one candidate model using per-origin and aggregate MAE.
   **Progressive hint:** At each origin, fit using timestamps at or before that origin and score only the next horizon. Preserve origin in the result table.
5. **Prediction intervals:** Produce forecast intervals and evaluate empirical coverage and width across rolling origins. Explain why a narrow interval is not useful when it misses too often.
   **Progressive hint:** For a nominal 90% interval, count actuals between lower and upper bounds and report support plus average width by horizon.
6. **Timestamp/data-quality debugging:** Validate a series containing duplicate timestamps, missing periods, an irregular interval, and a timezone transition before modeling.
   **Progressive hint:** Sort, assert monotonic unique timestamps, infer/declare frequency, and decide aggregation or imputation from domain meaning.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Rolling-origin evaluation


# Practice 5 — Prediction intervals


# Practice 6 — Timestamp/data-quality debugging
